In [58]:
import requests
import json
import pandas as pd

def fetch_all_images():
    batch_size = 1000
    page = 1
    all_images = []

    while True:
        url = (
            "https://rangers.urbanrivers.org/api/cameratrap/getCameratrapImages?"
            "species=Canis+latrans%2CCanis+familiaris%2CFelis+catus%2CCastor+canadensis%2COndatra+zibethicus%2C"
            "Sylvilagus+floridanus%2CSciurus+carolinensis%2CProcyon+lotor%2CLontra+canadensis%2CDidelphis+virginiana%2C"
            "Anas+platyrhynchos%2CBranta+canadensis%2CTrachemys+scripta+elegans%2CChelydra+serpentina%2CChrysemys+picta%2C"
            "Apalone+spinifera%2CColumba+livia%2CSturnus+vulgaris%2CAgelaius+phoeniceus%2CPasser+domesticus%2CTurdus+migratorius%2C"
            "Corvus+brachyrhynchos%2CArdea+herodias%2CNycticorax+nycticorax%2CAstur+cooperii%2CActitis+macularius%2CAix+sponsa%2C"
            "Ardea+alba%2CCardinalis+cardinalis%2CCyprinus+carpio&limit={}&page={}"
        ).format(batch_size, page)

        response = requests.get(url)
        if response.status_code != 200:
            print(f"Request failed at page {page} with status code {response.status_code}")
            break

        data = response.json()
        images = data.get("images", [])
        if not images:
            # No more images left
            break

        all_images.extend(images)
        print(f"Page {page}: Retrieved {len(images)} images")
        page += 1

    return all_images


def save_json(data, filename):
    with open(filename, "w") as f:
        json.dump(data, f)


def load_json(filename):
    with open(filename, "r") as f:
        return json.load(f)


def process_images(images):
    records = []
    for image in images:
        media_id = image.get("mediaID")
        public_url = image.get("publicURL")
        for species in image.get("speciesConsensus", []):
            scientific_name = species.get("scientificName")
            observation_count = species.get("observationCount")
            records.append({
                "mediaID": media_id,
                "publicURL": public_url,
                "scientificName": scientific_name,
                "observationCount": observation_count
            })
    df = pd.DataFrame(records)
    with pd.option_context('display.width', 0, 'display.max_colwidth', None):
        display(df.head(10))
    return df


if __name__ == "__main__":
    # Fetch and save JSON once
    images = fetch_all_images()
    save_json(images, "all_images.json")

    # Later: load JSON and process
    loaded_images = load_json("all_images.json")
    df = process_images(loaded_images)


Page 1: Retrieved 1000 images
Page 2: Retrieved 1000 images
Page 3: Retrieved 1000 images
Page 4: Retrieved 1000 images
Page 5: Retrieved 1000 images
Page 6: Retrieved 1000 images
Page 7: Retrieved 1000 images
Page 8: Retrieved 1000 images
Page 9: Retrieved 1000 images
Page 10: Retrieved 1000 images
Page 11: Retrieved 1000 images
Page 12: Retrieved 1000 images
Page 13: Retrieved 88 images


,mediaID,publicURL,scientificName,observationCount
0,9189635ca915cb3507c37f0b4c529846,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0158.JPG,Canis familiaris,1
1,9189635ca915cb3507c37f0b4c529846,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0158.JPG,None,2
2,9a6f3bbe7d62565c2ce5b632c0dfad55,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0160.JPG,Canis familiaris,1
3,9a6f3bbe7d62565c2ce5b632c0dfad55,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0160.JPG,None,2
4,ae560a001909c62e2993a1d2aa09c182,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0170.JPG,Canis familiaris,1
5,ae560a001909c62e2993a1d2aa09c182,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0170.JPG,Canis,1
6,ae560a001909c62e2993a1d2aa09c182,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0170.JPG,None,2
7,147f7b1eee7de62e93e1ddc8a9a049ae,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0182.JPG,None,1
8,147f7b1eee7de62e93e1ddc8a9a049ae,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0182.JPG,None,1
9,147f7b1eee7de62e93e1ddc8a9a049ae,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0182.JPG,Anas platyrhynchos,1


In [57]:
# seems like we have duplicates for every image and observation - just want a validated list
df2 = df.drop_duplicates().reset_index(drop=True)
with pd.option_context('display.width', 0, 'display.max_colwidth', None):
    display(df2.head(10))

print(len(df2))

,mediaID,publicURL,scientificName,observationCount
0,9189635ca915cb3507c37f0b4c529846,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0158.JPG,Canis familiaris,1
1,9189635ca915cb3507c37f0b4c529846,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0158.JPG,None,2
2,9a6f3bbe7d62565c2ce5b632c0dfad55,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0160.JPG,Canis familiaris,1
3,9a6f3bbe7d62565c2ce5b632c0dfad55,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0160.JPG,None,2
4,ae560a001909c62e2993a1d2aa09c182,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0170.JPG,Canis familiaris,1
5,ae560a001909c62e2993a1d2aa09c182,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0170.JPG,Canis,1
6,ae560a001909c62e2993a1d2aa09c182,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0170.JPG,None,2
7,147f7b1eee7de62e93e1ddc8a9a049ae,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0182.JPG,None,1
8,147f7b1eee7de62e93e1ddc8a9a049ae,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0182.JPG,Anas platyrhynchos,1
9,5e7e4ebe0b818ef91e3cd36d9a08515d,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0184.JPG,Branta canadensis,2


17329


In [72]:
# Filter to species with at least 2 votes
df3 = df2[(df2['observationCount'] > 1) & (df2['scientificName'].notna())]
df3 = df3.drop(columns=['scientificName_clean']).reset_index(drop=True)
with pd.option_context('display.width', 0, 'display.max_colwidth', None):
    display(df3.head())

,mediaID,publicURL,scientificName,observationCount
0,5e7e4ebe0b818ef91e3cd36d9a08515d,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0184.JPG,Branta canadensis,2
1,e27c658391c80374d29df456bb956f3a,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-31_LearningPlatformBeaver/DCIM/100MEDIA/SYFW0087.JPG,Canis familiaris,2
2,60d41330ba78a3f3dc18e6d87723d047,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-31_LearningPlatformBeaver/DCIM/100MEDIA/SYFW0143.JPG,Branta canadensis,2
3,00f357f17168f70cf929ad29d98c4610,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-31_LearningPlatformBeaver/DCIM/100MEDIA/SYFW0164.JPG,Canis familiaris,2
4,5d44b663d1e1de3b343df1eb9e69072e,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-31_LearningPlatformBeaver/DCIM/100MEDIA/SYFW0172.JPG,Canis familiaris,2
